In [1]:
# classification
# https://grok.com/chat/94d35e69-f6b6-4402-9107-14616814fa5e

import warnings
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score

import torch
from torchvision import transforms
import torchvision.transforms.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import ViTForImageClassification

/root/ml/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
warnings.filterwarnings("ignore")

### Params

In [3]:
DATSET_FILE = "../../data/oid/parquets/train_remote.parquet"

DEVICE = "cuda:0"
BATCH_SIZE = 8
TRAIN_HEAD_ONLY = False
LR = 1e-4

START_EPOCH = 0
NUM_EPOCHS = 10
TQDM_ITERS = 100
TQDM_INTERVAL = 60

TEST_RUN = False
VAL_SIZE = 0.1
PRECISION = 3
N_WORKERS = 4
SAVE_INTERVAL = 5

MODEL_NAME = "google/vit-base-patch16-224"

### Dataset

In [4]:
class ClassificationDataset(Dataset):
    def __init__(self, df, device):
        self.df = df
        self.transform = transforms.Compose(
            [
                transforms.ToTensor(),
            ]
        )
        self.device = device

    def __len__(self):
        return self.df.shape[0]

    def resize(self, image: Image.Image, size: int = 224) -> Image.Image:
        return F.resize(image, (size, size))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = self.resize(Image.open(row["image_path"]))
        image = self.transform(image.convert("RGB"))
        labels = torch.tensor([row["class"]], dtype=torch.int64)

        return image, labels

### Metrics

In [5]:
def eval_classification(model, dataloader):
    labels_true, labels_pred = [], []

    model.eval()
    with torch.no_grad():
        for images, labels in dataloader:
            images = torch.stack([image.to(DEVICE) for image in images])
            labels_true += torch.stack(labels).squeeze(1).tolist()

            outputs = model(images).logits

            _, raw_predict = torch.max(outputs, 1)
            labels_pred += raw_predict.to("cpu").tolist()

    metrics = {}
    metrics["precision"] = precision_score(
        labels_true, labels_pred, average="macro", zero_division=0
    )
    metrics["recall"] = recall_score(
        labels_true, labels_pred, average="macro", zero_division=0
    )
    metrics["f1"] = f1_score(labels_true, labels_pred, average="macro", zero_division=0)

    return metrics

### Train

In [6]:
df = pd.read_parquet(DATSET_FILE)
train_df, val_df = train_test_split(df, test_size=VAL_SIZE, random_state=0)
train_dataset, val_dataset = ClassificationDataset(
    train_df, DEVICE
), ClassificationDataset(val_df, DEVICE)
num_classes = df["class"].max() + 1

In [7]:
def collate_fn(batch):
    return tuple(zip(*batch))


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=N_WORKERS,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=N_WORKERS,
    pin_memory=True,
)

In [8]:
model = ViTForImageClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    ignore_mismatched_sizes=True,
)

model = model.to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
criterion = torch.nn.CrossEntropyLoss()

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([423]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([423, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
for epoch in range(NUM_EPOCHS):
    sum_loss, best_score = 0.0, -np.inf

    model.train()
    for images, labels in tqdm(train_loader):
        images = torch.stack([image.to(DEVICE) for image in images])
        labels = torch.stack([label.to(DEVICE) for label in labels])

        optimizer.zero_grad()
        outputs = model(images).logits

        loss = criterion(outputs, labels.squeeze())
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        _, predicted = torch.max(outputs, 1)

    epoch_loss = sum_loss / len(train_loader)

    metrics = eval_classification(model, val_loader)
    print(
        f"Epoch #{epoch+1} "
        + f"Loss: {epoch_loss:.3f} "
        + f"Precision: {metrics['precision']:.3f} "
        + f"Recall: {metrics['recall']:.3f} "
        + f"F1: {metrics['f1']:.3f}"
    )

    if metrics["f1"] > best_score:
        best_score = metrics["f1"]
        torch.save(model.state_dict(), f"models/vit_best.pth")

100%|██████████| 748/748 [00:30<00:00, 24.43it/s]


Epoch #1 Loss: 2.314 Precision: 0.215 Recall: 0.258 F1: 0.217


100%|██████████| 748/748 [00:30<00:00, 24.46it/s]


Epoch #2 Loss: 1.266 Precision: 0.257 Recall: 0.266 F1: 0.241


100%|██████████| 748/748 [00:30<00:00, 24.71it/s]


...

Epoch #8 Loss: 0.102 Precision: 0.225 Recall: 0.236 F1: 0.215


100%|██████████| 748/748 [00:30<00:00, 24.62it/s]


Epoch #9 Loss: 0.080 Precision: 0.291 Recall: 0.280 F1: 0.260


100%|██████████| 748/748 [00:30<00:00, 24.69it/s]


Epoch #10 Loss: 0.053 Precision: 0.266 Recall: 0.244 F1: 0.231
